In [ ]:
#GROUP1 = ["pigs051119","pigs061119","pigs071119","pigs081119","pigs091119",
#          "pigs151119","pigs161119","pigs271119","pigs281119"]                 # Train("pigs051119","pigs061119","pigs071119","pigs081119")
#GROUP2 = ["pigs291119","pigs301119","pigs011219","pigs021219a","pigs021219b"]  # Val
#GROUP3 = ["pigs031219","pigs041219","pigs051219","pigs071219","pigs081219",
#          "pigs091219","pigs101219a","pigs101219b","pigs111219"]               # Test("pigs051219","pigs071219")



In [ ]:
from pathlib import Path

BASE_UNZIPPED = Path("/content/drive/MyDrive/pig_data_unzipped")

GLOBAL_BG_PATH   = BASE_UNZIPPED / "background.png"
GLOBAL_MASK_PATH = BASE_UNZIPPED / "mask.png"

GROUP_DAYS = ["pigs051219","pigs071219"]
SPLIT_NAME = "test"

# ====== OUTPUT ======
OUT_ROOT    = Path(f"/content/drive/MyDrive/pig-selected_{SPLIT_NAME}")
OUT_IMG_DIR = OUT_ROOT / f"images_{SPLIT_NAME}"
POOL_CSV    = OUT_ROOT / f"pool_{SPLIT_NAME}.csv"
SELECT_CSV  = OUT_ROOT / f"selected_{SPLIT_NAME}.csv"
MANIFEST    = OUT_ROOT / f"manifest_{SPLIT_NAME}.csv"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

TARGET_N = 400

WINDOW_SEC      = 1.0
TOPK_PER_WINDOW = 3

PHASH_HAM_PASS1         = 5
PHASH_HAM_PASS2         = 4
PHASH_RECENT_WIN        = 100
MIN_GAP_LEAF_PASS1      = 3.0
MIN_GAP_LEAF_PASS2      = 2.0
MIN_GAP_LEAF_PASS3      = 1.0   # pass fill
PER_LEAF_SOFT_FACTOR_1  = 2.5

print("BASE_UNZIPPED:", BASE_UNZIPPED)
print("OUT_ROOT     :", OUT_ROOT)


In [ ]:
import os, csv, cv2, numpy as np
from pathlib import Path
from collections import defaultdict

# ---------- data helpers ----------
def validate_day_dir(day_path: Path) -> bool:
    try:
        pigs_dirs = [p for p in day_path.iterdir() if p.is_dir() and p.name.upper().startswith("PIGS")]
        for pd in pigs_dirs:
            leaves = [q for q in pd.iterdir() if q.is_dir() and q.name.isdigit()]
            for lf in leaves:
                if (lf/"color.mp4").exists():
                    return True
    except Exception:
        pass
    return False

def list_day_dirs(base: Path):
    days = [p for p in base.iterdir() if p.is_dir() and p.name.lower().startswith("pigs")]
    days = [p for p in days if validate_day_dir(p)]
    days.sort(key=lambda p: p.name)
    return days

def find_inner_dataset_root(day_root: Path):
    cand = [p for p in day_root.iterdir() if p.is_dir() and p.name.upper().startswith("PIGS")]
    return cand[0] if cand else day_root

def iter_leaf_dirs(day_inner_root: Path):
    leaves = [p for p in day_inner_root.iterdir() if p.is_dir() and p.name.isdigit()]
    leaves.sort(key=lambda p: int(p.name))
    return leaves

def read_times_txt(txt_path: Path):
    ts = []
    try:
        with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
            for ln in f:
                ln = ln.strip()
                if not ln: continue
                parts = ln.replace(",", ".").split()
                ts.append(float(parts[-1]))
        if len(ts) >= 2:
            diffs = np.diff(ts); med = float(np.median(diffs))
            if 0 < med < 1e-3: ts = [t/1e6 for t in ts]
            elif med >= 1000: ts = [t/1000.0 for t in ts]    # ms -> s
        return ts if ts else None
    except Exception:
        return None

# ---------- image & score ----------
def load_bg_and_mask_fixed():
    bg = None
    if GLOBAL_BG_PATH.exists():
        bg = cv2.imdecode(np.fromfile(str(GLOBAL_BG_PATH), dtype=np.uint8), cv2.IMREAD_COLOR)
    mask = None
    if GLOBAL_MASK_PATH.exists():
        m = cv2.imdecode(np.fromfile(str(GLOBAL_MASK_PATH), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
        if m is not None:
            _, m = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
            mask = m
    return bg, mask

def resize_to(img, W, H, is_mask=False):
    if img is None: return None
    inter = cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
    if img.shape[:2] != (H, W):
        return cv2.resize(img, (W, H), interpolation=inter)
    return img

def down_gray(bgr, size=(160,90)):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    return cv2.resize(g, size, interpolation=cv2.INTER_AREA)

def activity_score(curr_small, prev_small=None, bg_small=None):
    vals = []
    if prev_small is not None:
        vals.append(np.mean(np.abs(curr_small.astype(np.float32)-prev_small.astype(np.float32))))
    if bg_small is not None:
        vals.append(np.mean(np.abs(curr_small.astype(np.float32)-bg_small.astype(np.float32))))
    return max(vals) if vals else 0.0

# ---------- aHash & Hamming ----------
def ahash64(gray, size=8):
    small = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA)
    avg = small.mean()
    bits = (small > avg).astype(np.uint8).flatten()
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return h

def hamming64(a, b):
    return bin(a ^ b).count("1")

# ---------- CSV inspect & fix ----------
def inspect_csv_header(csv_path: Path, peek=2):
    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.reader(f)
        rows = list(rd)
    if not rows:
        print("[POOL] empty"); return
    header = rows[0]
    print("[Header]", header)
    print("[Peek]", rows[1:1+peek])

def fix_pool_csv_if_needed(pool_csv: Path) -> Path:
    """
      0: day, 1: inner, 2: leaf, 3: frame_idx, 4: activity, 5: time_s, 6: ahash,
      7: width, 8: height, 9: fps, 10: k_in_win
    """
    if not pool_csv.exists():
        raise FileNotFoundError(f"pool not found: {pool_csv}")
    with open(pool_csv, "r", encoding="utf-8") as f:
        header = f.readline().strip().split(",")
    must = {"day","inner","leaf","frame_idx","time_s","ahash"}
    if must.issubset(set(header)):
        print("[FIX] Pool header OK:", pool_csv.name)
        return pool_csv

    fixed = pool_csv.with_name(pool_csv.stem + "_fixed.csv")
    with open(pool_csv, "r", encoding="utf-8") as fin, \
         open(fixed, "w", newline="", encoding="utf-8") as fout:
        reader = csv.reader(fin)
        writer = csv.DictWriter(fout, fieldnames=[
            "day","inner","leaf","frame_idx","time_s","activity","ahash","width","height","fps","k_in_win"
        ])
        writer.writeheader()
        for row in reader:
            if not row or len(row) < 11:
                continue
            try:
                writer.writerow({
                    "day":       row[0],
                    "inner":     row[1],
                    "leaf":      row[2],
                    "frame_idx": int(float(row[3])),
                    "time_s":    f"{float(row[5]):.3f}",
                    "activity":  f"{float(row[4]):.3f}",
                    "ahash":     str(int(float(row[6]))),
                    "width":     int(float(row[7])),
                    "height":    int(float(row[8])),
                    "fps":       f"{float(row[9]):.3f}",
                    "k_in_win":  int(float(row[10])),
                })
            except:
                pass
    print("[FIX] Wrote:", fixed)
    return fixed


In [ ]:
from tqdm.auto import tqdm

def build_candidate_pool_topk_clean(days_names, pool_csv_path: Path, window_sec=1.0, topk=3, mode="w"):
    """
    """
    all_days = list_day_dirs(BASE_UNZIPPED)
    name2path = {p.name: p for p in all_days}
    days_list = [name2path[n] for n in days_names if n in name2path]
    if not days_list:
        raise RuntimeError()

    bg_fixed, mask_fixed = load_bg_and_mask_fixed()
    if bg_fixed is None:
        pass
    if mask_fixed is None:
        pass

    pool_csv_path.parent.mkdir(parents=True, exist_ok=True)
    need_header = (mode == "w") or (not pool_csv_path.exists())
    mf = open(pool_csv_path, mode, newline="", encoding="utf-8")
    writer = csv.DictWriter(mf, fieldnames=[
        "day","inner","leaf","frame_idx","time_s","activity","ahash","width","height","fps","k_in_win"
    ])
    if need_header:
        writer.writeheader()

    day_bar = tqdm(days_list, desc="Phase A: days", unit="day", position=0)
    for day_root in day_bar:
        inner_root = find_inner_dataset_root(day_root)
        leaves = iter_leaf_dirs(inner_root)
        leaf_bar = tqdm(leaves, desc=f"{day_root.name}", unit="leaf", position=1, leave=False)

        for leaf in leaf_bar:
            color_mp4 = leaf / "color.mp4"
            times_txt = leaf / "times.txt"
            if not color_mp4.exists():
                continue

            cap = cv2.VideoCapture(str(color_mp4))
            fps = cap.get(cv2.CAP_PROP_FPS) or 6.0
            n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1280)
            H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)
            ts = read_times_txt(times_txt) if times_txt.exists() else None

            mask = resize_to(mask_fixed, W, H, is_mask=True) if mask_fixed is not None else None
            bg   = resize_to(bg_fixed,   W, H) if bg_fixed is not None else None
            if bg is not None and mask is not None:
                bg = cv2.bitwise_and(bg, bg, mask=mask)
            bg_small = down_gray(bg) if bg is not None else None

            frm_bar = tqdm(total=n_frames, desc=f"{leaf.name}", unit="frm", position=2, leave=False)
            last_small = None
            win_start, win_end = 0.0, window_sec
            win_cands = []  # (act, idx, time_s, ahash)

            i = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break

                if ts is not None and i < len(ts):
                    time_s = float(ts[i] - ts[0])
                else:
                    time_s = float(i) / max(fps, 1e-6)

                # ROI
                if mask is not None:
                    frame = cv2.bitwise_and(frame, frame, mask=mask)

                cur_small = down_gray(frame)
                act = activity_score(cur_small, prev_small=last_small, bg_small=bg_small)
                gray32 = cv2.resize(cur_small, (32,32), interpolation=cv2.INTER_AREA)
                ah = ahash64(gray32, size=8)

                win_cands.append((act, i, time_s, ah))
                if len(win_cands) > topk:
                    win_cands.sort(key=lambda x: x[0], reverse=True)
                    win_cands = win_cands[:topk]

                if time_s >= win_end:
                    win_cands.sort(key=lambda x: x[2])
                    for rank, (a, idx, ts_s, ahv) in enumerate(win_cands, start=1):
                        writer.writerow({
                            "day": day_root.name, "inner": inner_root.name, "leaf": leaf.name,
                            "frame_idx": idx, "time_s": f"{ts_s:.3f}", "activity": f"{a:.3f}",
                            "ahash": str(ahv), "width": W, "height": H, "fps": f"{fps:.3f}", "k_in_win": rank
                        })
                    while time_s >= win_end:
                        win_start += window_sec
                        win_end   += window_sec
                    win_cands = []

                last_small = cur_small
                i += 1
                frm_bar.update(1)

            if win_cands:
                win_cands.sort(key=lambda x: x[2])
                for rank, (a, idx, ts_s, ahv) in enumerate(win_cands, start=1):
                    writer.writerow({
                        "day": day_root.name, "inner": inner_root.name, "leaf": leaf.name,
                        "frame_idx": idx, "time_s": f"{ts_s:.3f}", "activity": f"{a:.3f}",
                        "ahash": str(ahv), "width": W, "height": H, "fps": f"{fps:.3f}", "k_in_win": rank
                    })

            frm_bar.close()
            cap.release()

        leaf_bar.close()
    day_bar.close()
    mf.close()
    print(f"[Phase A] DONE. Pool CSV: {pool_csv_path}")


In [ ]:
# import os;
# if os.path.exists(POOL_CSV): os.remove(POOL_CSV)
build_candidate_pool_topk_clean(GROUP_DAYS, POOL_CSV, window_sec=WINDOW_SEC, topk=TOPK_PER_WINDOW, mode="w")


In [ ]:
import csv, numpy as np
from collections import defaultdict

def load_pool_records(pool_csv: Path):
    pool_for_read = fix_pool_csv_if_needed(pool_csv)
    recs = []  # (day, inner, leaf, frame_idx, time_s, activity, ahash)
    with open(pool_for_read, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            try:
                day, inner, leaf = r["day"], r["inner"], r["leaf"]
                fi   = int(float(r["frame_idx"]))
                t_s  = float(r["time_s"])
                act  = float(r.get("activity","0"))
                ah   = int(r["ahash"])
                recs.append((day, inner, leaf, fi, t_s, act, ah))
            except:
                pass
    groups = defaultdict(list)
    order_days, seen = [], set()
    for day, inner, leaf, fi, t_s, act, ah in recs:
        key = (day, inner, leaf)
        groups[key].append({"frame_idx":fi, "time_s":t_s, "activity":act, "ahash":ah})
        if day not in seen:
            order_days.append(day); seen.add(day)
    for k in groups:
        groups[k].sort(key=lambda d: d["time_s"])
    all_leaves = sorted(groups.keys(), key=lambda k: (order_days.index(k[0]) if k[0] in order_days else 9999, int(k[2])))
    return groups, all_leaves

def select_multipass_non_destructive(
    pool_csv: Path, target_n: int,
    phash_thr_pass1=5, phash_thr_pass2=4, recent_win=100,
    min_gap_leaf_pass1=3.0, min_gap_leaf_pass2=2.0, min_gap_leaf_pass3=1.0,
    per_leaf_soft_factor_1=2.5,
    out_select_csv: Path=None,
    force_fill=True
):
    groups, all_leaves = load_pool_records(pool_csv)
    if not all_leaves:
        raise RuntimeError()

    selected = []          # (day, inner, leaf, fi, t_s, act, ah)
    selected_hashes = []
    leaf_kept = {k:0 for k in all_leaves}
    leaf_last_t = {k:-1e9 for k in all_leaves}
    used = {k: np.zeros(len(groups[k]), dtype=bool) for k in all_leaves}

    def _try_round(thr_ham, min_gap, use_soft_cap):
        nonlocal selected, selected_hashes
        progressed = True
        per_leaf_soft_cap = None
        if use_soft_cap:
            n_nonempty = sum(1 for k in all_leaves if not used[k].all()) or 1
            per_leaf_soft_cap = int(np.ceil(target_n / n_nonempty * per_leaf_soft_factor_1))
            per_leaf_soft_cap = max(10, per_leaf_soft_cap)

        pos = {k:0 for k in all_leaves}

        while len(selected) < target_n and progressed:
            progressed = False
            for k in all_leaves:
                if len(selected) >= target_n: break
                if use_soft_cap and per_leaf_soft_cap is not None and leaf_kept[k] >= per_leaf_soft_cap:
                    continue
                g = groups[k]; n = len(g)
                i = pos[k]
                found = False
                while i < n:
                    if used[k][i]:
                        i += 1; continue
                    cand = g[i]
                    ts, fi, act, ah = cand["time_s"], cand["frame_idx"], cand["activity"], cand["ahash"]
                    if (ts - leaf_last_t[k]) < min_gap:
                        i += 1; continue
                    if thr_ham is not None:
                        if any(hamming64(ah, rh) < thr_ham for rh in selected_hashes[-recent_win:]):
                            i += 1; continue
                    used[k][i] = True
                    selected.append((k[0], k[1], k[2], fi, ts, act, ah))
                    selected_hashes.append(ah)
                    leaf_kept[k] += 1
                    leaf_last_t[k] = ts
                    pos[k] = i + 1
                    progressed = True
                    found = True
                    break
                if not found:
                    pos[k] = i
        return progressed

    _try_round(PHASH_HAM_PASS1, MIN_GAP_LEAF_PASS1, use_soft_cap=True)
    if len(selected) < target_n:
        _try_round(PHASH_HAM_PASS2, MIN_GAP_LEAF_PASS2, use_soft_cap=False)
    if len(selected) < target_n:
        _try_round(None, MIN_GAP_LEAF_PASS3, use_soft_cap=False)

    if force_fill and len(selected) < target_n:
        need = target_n - len(selected)
        print(f"[Pass4] emergency fill need {need}")
        remain = {k: [i for i, u in enumerate(used[k]) if not u] for k in all_leaves}
        rp = {k:0 for k in all_leaves}
        done = False
        while not done and len(selected) < target_n:
            progressed = False
            for k in all_leaves:
                if len(selected) >= target_n: break
                arr, j = remain[k], rp[k]
                if j >= len(arr): continue
                i = arr[j]
                rp[k] = j + 1
                cand = groups[k][i]
                ts, fi, act, ah = cand["time_s"], cand["frame_idx"], cand["activity"], cand["ahash"]
                used[k][i] = True
                selected.append((k[0], k[1], k[2], fi, ts, act, ah))
                selected_hashes.append(ah)
                leaf_kept[k] += 1
                progressed = True
                if len(selected) >= target_n: break
            if not progressed:
                done = True

    # ghi selected.csv
    if out_select_csv is not None:
        out_select_csv.parent.mkdir(parents=True, exist_ok=True)
        with open(out_select_csv, "w", newline="", encoding="utf-8") as f:
            wr = csv.writer(f)
            wr.writerow(["day","inner","leaf","frame_idx","time_s","activity","ahash"])
            for day,inner,leaf,fi,ts,act,ah in selected:
                wr.writerow([day,inner,leaf,fi,f"{ts:.3f}",f"{act:.3f}",str(ah)])

    print(f"[Phase B.1] Selected {len(selected)} / target {target_n} (non-destructive multipass).")
    return selected


In [ ]:
selected = select_multipass_non_destructive(
    POOL_CSV, target_n=TARGET_N,
    phash_thr_pass1=PHASH_HAM_PASS1, phash_thr_pass2=PHASH_HAM_PASS2, recent_win=PHASH_RECENT_WIN,
    min_gap_leaf_pass1=MIN_GAP_LEAF_PASS1, min_gap_leaf_pass2=MIN_GAP_LEAF_PASS2, min_gap_leaf_pass3=MIN_GAP_LEAF_PASS3,
    per_leaf_soft_factor_1=PER_LEAF_SOFT_FACTOR_1,
    out_select_csv=SELECT_CSV,
    force_fill=True
)


## Archived Experiment Note

This notebook text was normalized from corrupted non-English notes. The code cell order is preserved; use the production package and docs for the maintained workflow.


In [ ]:
from tqdm.auto import tqdm
from itertools import groupby

def extract_selected_images_with_mask(selected_list, out_img_dir: Path, manifest_csv: Path):
    out_img_dir.mkdir(parents=True, exist_ok=True)
    # map day->path
    all_days = list_day_dirs(BASE_UNZIPPED)
    name2path = {p.name: p for p in all_days}

    # manifest
    mf = open(manifest_csv, "w", newline="", encoding="utf-8")
    writer = csv.DictWriter(mf, fieldnames=[
        "global_index","image","day","inner","leaf","frame_idx","frame_time_s","activity"
    ])
    writer.writeheader()

    selected_list.sort(key=lambda x: (x[0], int(x[2]), x[4]))
    gi = 1

    def key_fn(rec): return (rec[0], rec[1], rec[2])  # (day, inner, leaf)
    unique_keys = list({(d,i,l) for d,i,l,_,_,_,_ in selected_list})

    _, mask_fixed = load_bg_and_mask_fixed()

    for (day, inner, leaf), block in tqdm(
        groupby(selected_list, key=key_fn), total=len(unique_keys),
        desc="Phase B.2: extract", unit="leaf"
    ):
        day_root = name2path.get(day, None)
        if day_root is None:
            print(f"[WARN] day not found: {day}");
            continue
        inner_root = day_root / inner
        leaf_dir   = inner_root / leaf
        color_mp4  = leaf_dir / "color.mp4"
        if not color_mp4.exists():
            print(f"[WARN] missing color.mp4: {leaf_dir}")
            continue

        cap = cv2.VideoCapture(str(color_mp4))
        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1280)
        H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)
        mask = resize_to(mask_fixed, W, H, is_mask=True) if mask_fixed is not None else None

        for rec in block:
            _,_,_, frame_idx, time_s, act, _ = rec
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, frame = cap.read()
            if not ok or frame is None:
                continue
            if mask is not None:
                frame = cv2.bitwise_and(frame, frame, mask=mask)

            fname = f"{gi:06d}.jpg"
            fpath = out_img_dir / fname
            ok2, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
            if ok2:
                buf.tofile(str(fpath))
                writer.writerow({
                    "global_index": gi, "image": fname,
                    "day": day, "inner": inner, "leaf": leaf,
                    "frame_idx": int(frame_idx),
                    "frame_time_s": f"{float(time_s):.3f}",
                    "activity": f"{float(act):.3f}",
                })
                gi += 1

        cap.release()

    mf.close()
    print(f"[Phase B.2] Wrote {gi-1} images -> {out_img_dir}")
    print(f"[Phase B.2] Manifest:", manifest_csv)


In [ ]:
extract_selected_images_with_mask(selected, OUT_IMG_DIR, MANIFEST)